# Posterior Hot Zone EEG — Exploration

Walk through the analysis pipeline on a single subject from Sleep-EDF Expanded. Once this works end-to-end, run the full pipeline across subjects with `scripts/run_analysis.py`.

Steps mirror the proposal:
1. Load PSG + hypnogram
2. Bandpass filter (0.3–40 Hz)
3. Epoch into 30-second chunks aligned to sleep stages
4. Keep N2, N3, REM
5. Compute band power per epoch via Welch's method
6. Aggregate per stage × channel × band
7. Posterior − frontal contrast
8. Plots

In [ ]:
import sys
from pathlib import Path

# Make the src/ package importable when running this notebook from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_loader import download_sleep_edf, load_raw_with_annotations
from src.preprocessing import filter_raw, make_epochs, select_channels, keep_sleep_stages
from src.features import (
    compute_band_powers,
    aggregate_subject,
    posterior_frontal_contrast,
    mean_spectra_by_stage,
)
from src.plots import plot_spectra, plot_bandpower_boxes, plot_contrast_bars

%matplotlib inline

## 1. Download one subject

In [ ]:
SUBJECT_ID = 0
DATA_DIR = Path('../data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

pairs = download_sleep_edf([SUBJECT_ID], recording=1, path=DATA_DIR)
psg_path, hyp_path = pairs[0]
print('PSG:', psg_path)
print('HYP:', hyp_path)

## 2. Load and filter

In [ ]:
raw = load_raw_with_annotations(psg_path, hyp_path)
print(raw.info)
filter_raw(raw, l_freq=0.3, h_freq=40.0)

## 3. Epoch and select stages

Builds 30-second epochs from hypnogram annotations, keeps the two EEG channels, and drops Wake / N1.

In [ ]:
epochs = make_epochs(raw)
epochs = select_channels(epochs)
epochs = keep_sleep_stages(epochs)
print('Epochs per stage:')
for stage, code in epochs.event_id.items():
    n = int((epochs.events[:, 2] == code).sum())
    print(f'  {stage}: {n}')

## 4. Band powers

In [ ]:
band_df = compute_band_powers(epochs)
subject_df = aggregate_subject(band_df, SUBJECT_ID)
subject_df

## 5. Posterior − frontal contrast

In [ ]:
contrast = posterior_frontal_contrast(subject_df)
contrast

## 6. Plots

With just one subject these are exploratory — group-level plots come from `scripts/run_analysis.py` once multiple subjects are aggregated.

In [ ]:
spectra, freqs, channels = mean_spectra_by_stage(epochs)
plot_spectra(spectra, freqs, channels)
plt.show()

In [ ]:
plot_bandpower_boxes(subject_df)
plt.show()

In [ ]:
plot_contrast_bars(contrast)
plt.show()